# Bloque 6 - Cierre del proyecto
## Proyecto: Red Social Ficticia - Informe final guiado

Este último bloque **no introduce contenidos nuevos**. Es un cierre breve para demostrar que sabes recorrer un flujo completo de análisis de datos en Python:

1. cargar las fuentes originales,
2. limpiar los datos necesarios,
3. construir un dataset de análisis,
4. calcular una métrica de engagement,
5. responder tres preguntas analíticas,
6. crear tres gráficos interactivos,
7. escribir una conclusión breve.

> Este cuaderno parte solo de `../data/usuarios.csv`, `../data/publicaciones.xlsx` y `../data/conexiones.json`. No depende de los ficheros generados en los bloques anteriores.

---
### Objetivos
- Consolidar la carga, auditoría y limpieza de datos.
- Combinar usuarios y publicaciones con Pandas.
- Calcular una métrica de engagement robusta.
- Responder preguntas sencillas con tablas resumen.
- Comunicar resultados mediante gráficos de Plotly.
- Cerrar el proyecto con una interpretación breve.

### Cómo trabajar este cuaderno
1. Ejecuta primero la celda de preparación.
2. Completa los **seis ejercicios básicos** en orden.
3. Sustituye cada `None` por tu código.
4. Al terminar, reinicia el kernel y ejecuta el cuaderno de arriba abajo.
5. Entrega el cuaderno ejecutado y los gráficos HTML generados.


## Preparación


In [ ]:
# Importamos las librerías necesarias y creamos la carpeta de salida.
# Esta celda ya está resuelta, no la modifiques.
import json
import os

import numpy as np
import pandas as pd
import plotly.express as px

os.makedirs('../output_graficos', exist_ok=True)


---
## Ejercicios básicos

Los ejemplos de los enunciados usan datos inventados y **nunca** son la solución.


In [ ]:
# Ejercicio 1 - Carga y auditoría inicial
# Los datos del proyecto están en tres fuentes: usuarios, publicaciones y
# conexiones. En este ejercicio los cargarás y harás una auditoría mínima.
#
# 1. Carga '../data/usuarios.csv' en 'df_usuarios'.
# 2. Carga la hoja 'publicaciones' de '../data/publicaciones.xlsx' en
#    'df_publicaciones'.
# 3. Carga '../data/conexiones.json' en 'conexiones'.
# 4. Guarda en 'auditoria_usuarios' y 'auditoria_publicaciones' una tabla con
#    una fila por columna y las columnas: 'columna', 'tipo_dato', 'n_nulos' y
#    'n_distintos'.
# 5. Guarda en 'resumen_carga' un diccionario con el número de filas de
#    usuarios, publicaciones y registros de conexiones.

df_usuarios = pd.read_csv('../data/usuarios.csv')
df_publicaciones = pd.read_excel('../data/publicaciones.xlsx', sheet_name='publicaciones')
with open('../data/conexiones.json', encoding='utf-8') as fichero:
    conexiones = json.load(fichero)


def auditar_tabla(df):
    return pd.DataFrame({
        'columna': df.columns,
        'tipo_dato': [str(tipo) for tipo in df.dtypes],
        'n_nulos': df.isna().sum().to_numpy(),
        'n_distintos': df.nunique(dropna=True).to_numpy(),
    })


auditoria_usuarios = auditar_tabla(df_usuarios)
auditoria_publicaciones = auditar_tabla(df_publicaciones)
resumen_carga = {
    'usuarios': len(df_usuarios),
    'publicaciones': len(df_publicaciones),
    'conexiones': len(conexiones),
}

print(resumen_carga)
display(auditoria_usuarios)
display(auditoria_publicaciones)


In [ ]:
# Ejercicio 2 - Limpieza mínima
# Prepara copias limpias de usuarios y publicaciones. No modifiques las
# tablas originales.
#
# 1. Guarda en 'usuarios_limpios' una copia de 'df_usuarios'. Rellena los
#    nulos de 'edad' con la mediana y los de 'ciudad' y 'bio' con
#    'Desconocido'.
# 2. Guarda en 'publicaciones_limpias' una copia de 'df_publicaciones'.
#    Rellena los nulos de 'likes' y 'comentarios' con la mediana de cada
#    columna.
# 3. Convierte 'fecha' a tipo fecha en 'publicaciones_limpias'.
# 4. Guarda en 'nulos_originales' el total de nulos de las tablas originales
#    y en 'nulos_limpios' el total de nulos de las tablas limpias.
# 5. Guarda en 'originales_intactos' si las tablas originales siguen teniendo
#    los mismos nulos que al principio.

nulos_originales = int(df_usuarios.isna().sum().sum() + df_publicaciones.isna().sum().sum())

usuarios_limpios = df_usuarios.copy()
usuarios_limpios['edad'] = usuarios_limpios['edad'].fillna(usuarios_limpios['edad'].median())
usuarios_limpios['ciudad'] = usuarios_limpios['ciudad'].fillna('Desconocido')
usuarios_limpios['bio'] = usuarios_limpios['bio'].fillna('Desconocido')

publicaciones_limpias = df_publicaciones.copy()
for columna in ['likes', 'comentarios']:
    publicaciones_limpias[columna] = publicaciones_limpias[columna].fillna(publicaciones_limpias[columna].median())
publicaciones_limpias['fecha'] = pd.to_datetime(publicaciones_limpias['fecha'])

nulos_limpios = int(usuarios_limpios.isna().sum().sum() + publicaciones_limpias.isna().sum().sum())
originales_intactos = nulos_originales == int(df_usuarios.isna().sum().sum() + df_publicaciones.isna().sum().sum())

print('Nulos originales:', nulos_originales)
print('Nulos tras limpiar:', nulos_limpios)
print('Originales intactos:', originales_intactos)


In [ ]:
# Ejercicio 3 - Dataset de análisis y engagement
# Combina publicaciones y usuarios para crear el dataset final del cierre.
#
# 1. Guarda en 'df_final' la unión de 'publicaciones_limpias' con
#    'usuarios_limpios' por 'id_usuario', conservando todas las publicaciones.
# 2. Añade a 'df_final' la columna 'engagement_rate' con la fórmula:
#       (likes + comentarios + compartidos) / visualizaciones
#    Si 'visualizaciones' es 0, el engagement debe valer 0.
# 3. Guarda en 'resumen_final' un diccionario con filas, columnas, nulos y
#    engagement medio.
# 4. Guarda 'df_final' en '../data/dataset_cierre.parquet' con compresión
#    snappy y sin índice.

df_final = pd.merge(
    publicaciones_limpias,
    usuarios_limpios,
    on='id_usuario',
    how='left',
    validate='many_to_one',
)

interacciones = df_final['likes'] + df_final['comentarios'] + df_final['compartidos']
visualizaciones = df_final['visualizaciones']
df_final['engagement_rate'] = (interacciones / visualizaciones.where(visualizaciones != 0)).fillna(0)

resumen_final = {
    'filas': len(df_final),
    'columnas': len(df_final.columns),
    'nulos': int(df_final.isna().sum().sum()),
    'engagement_medio': round(float(df_final['engagement_rate'].mean()), 4),
}

df_final.to_parquet('../data/dataset_cierre.parquet', compression='snappy', index=False)

print(resumen_final)
display(df_final.head())


In [ ]:
# Ejercicio 4 - Tres preguntas analíticas
# Responde tres preguntas sobre el dataset final usando tablas resumen.
#
# Pregunta 1: ¿qué ciudades tienen mayor engagement medio?
# Guarda la respuesta en 'engagement_por_ciudad' con las columnas:
# 'ciudad', 'publicaciones', 'engagement_medio' y 'seguidores_medios'.
#
# Pregunta 2: ¿qué tipo de contenido genera mayor engagement?
# Guarda la respuesta en 'engagement_por_contenido' con las columnas:
# 'tipo_contenido', 'publicaciones', 'engagement_medio' y 'likes_medios'.
#
# Pregunta 3: ¿hay relación entre seguidores y engagement?
# Guarda en 'engagement_por_usuario' una tabla con una fila por usuario y en
# 'correlacion_seguidores_engagement' la correlación entre 'seguidores' y
# 'engagement_promedio', redondeada a 3 decimales.

engagement_por_ciudad = (
    df_final.groupby('ciudad', as_index=False)
    .agg(
        publicaciones=('id_publicacion', 'count'),
        engagement_medio=('engagement_rate', 'mean'),
        seguidores_medios=('seguidores', 'mean'),
    )
    .sort_values('engagement_medio', ascending=False)
)
engagement_por_ciudad['engagement_medio'] = engagement_por_ciudad['engagement_medio'].round(4)
engagement_por_ciudad['seguidores_medios'] = engagement_por_ciudad['seguidores_medios'].round(1)

engagement_por_contenido = (
    df_final.groupby('tipo_contenido', as_index=False)
    .agg(
        publicaciones=('id_publicacion', 'count'),
        engagement_medio=('engagement_rate', 'mean'),
        likes_medios=('likes', 'mean'),
    )
    .sort_values('engagement_medio', ascending=False)
)
engagement_por_contenido['engagement_medio'] = engagement_por_contenido['engagement_medio'].round(4)
engagement_por_contenido['likes_medios'] = engagement_por_contenido['likes_medios'].round(1)

engagement_por_usuario = (
    df_final.groupby('id_usuario', as_index=False)
    .agg(
        engagement_promedio=('engagement_rate', 'mean'),
        n_publicaciones=('id_publicacion', 'count'),
    )
    .merge(usuarios_limpios[['id_usuario', 'nombre', 'tipo_cuenta', 'seguidores']], on='id_usuario')
)
correlacion_seguidores_engagement = round(
    float(engagement_por_usuario['seguidores'].corr(engagement_por_usuario['engagement_promedio'])),
    3,
)

display(engagement_por_ciudad)
display(engagement_por_contenido)
display(engagement_por_usuario.sort_values('engagement_promedio', ascending=False).head())
print('Correlación seguidores-engagement:', correlacion_seguidores_engagement)


In [ ]:
# Ejercicio 5 - Tres gráficos interactivos
# Crea un gráfico para cada pregunta analítica y expórtalo a HTML.
#
# 1. 'fig_ciudad': barras con el engagement medio por ciudad.
#    Exporta a '../output_graficos/engagement_por_ciudad.html'.
# 2. 'fig_contenido': barras con el engagement medio por tipo de contenido.
#    Exporta a '../output_graficos/engagement_por_tipo.html'.
# 3. 'fig_seguidores': dispersión de seguidores frente a engagement medio por
#    usuario, coloreada por tipo de cuenta.
#    Exporta a '../output_graficos/seguidores_vs_engagement.html'.
# 4. Guarda en 'graficos_generados' la lista de rutas exportadas.

top_ciudades = engagement_por_ciudad.head(10)
fig_ciudad = px.bar(
    top_ciudades,
    x='ciudad',
    y='engagement_medio',
    text='engagement_medio',
    title='Engagement medio por ciudad',
    labels={'ciudad': 'Ciudad', 'engagement_medio': 'Engagement medio'},
    template='plotly_white',
)
fig_ciudad.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_ciudad.update_layout(showlegend=False)

fig_contenido = px.bar(
    engagement_por_contenido,
    x='tipo_contenido',
    y='engagement_medio',
    text='engagement_medio',
    title='Engagement medio por tipo de contenido',
    labels={'tipo_contenido': 'Tipo de contenido', 'engagement_medio': 'Engagement medio'},
    template='plotly_white',
)
fig_contenido.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_contenido.update_layout(showlegend=False)

fig_seguidores = px.scatter(
    engagement_por_usuario,
    x='seguidores',
    y='engagement_promedio',
    color='tipo_cuenta',
    size='n_publicaciones',
    hover_name='nombre',
    title='Seguidores frente a engagement medio',
    labels={
        'seguidores': 'Seguidores',
        'engagement_promedio': 'Engagement medio',
        'tipo_cuenta': 'Tipo de cuenta',
        'n_publicaciones': 'Publicaciones',
    },
    template='plotly_white',
)

graficos_generados = [
    '../output_graficos/engagement_por_ciudad.html',
    '../output_graficos/engagement_por_tipo.html',
    '../output_graficos/seguidores_vs_engagement.html',
]
fig_ciudad.write_html(graficos_generados[0])
fig_contenido.write_html(graficos_generados[1])
fig_seguidores.write_html(graficos_generados[2])

print(graficos_generados)
fig_ciudad.show()
fig_contenido.show()
fig_seguidores.show()


In [ ]:
# Ejercicio 6 - Conclusión breve
# Escribe una conclusión breve del análisis. Debe tener entre 5 y 8 líneas y
# comentar, al menos:
#
# - qué ciudad destaca,
# - qué tipo de contenido obtiene más engagement,
# - si la relación entre seguidores y engagement parece fuerte o débil,
# - una limitación de los datos,
# - una mejora que harías con más tiempo.
#
# Guarda el texto en 'conclusion_breve'.

ciudad_destacada = engagement_por_ciudad.iloc[0]['ciudad']
contenido_destacado = engagement_por_contenido.iloc[0]['tipo_contenido']

if abs(correlacion_seguidores_engagement) >= 0.7:
    fuerza_correlacion = 'fuerte'
elif abs(correlacion_seguidores_engagement) >= 0.3:
    fuerza_correlacion = 'moderada'
else:
    fuerza_correlacion = 'débil'

conclusion_breve = '\n'.join([
    f'La ciudad con mayor engagement medio es {ciudad_destacada}.',
    f'El tipo de contenido que mejor funciona es {contenido_destacado}.',
    f'La relación entre seguidores y engagement parece {fuerza_correlacion} '
    f'(correlación = {correlacion_seguidores_engagement}).',
    'Los datos permiten comparar perfiles y publicaciones, pero son ficticios y tienen un tamaño limitado.',
    'Con más tiempo ampliaría el análisis temporal y comprobaría si los resultados se mantienen por mes y tipo de cuenta.',
])

print(conclusion_breve)


---
## Ampliación opcional

Si terminas el cierre básico y quieres profundizar, puedes elegir una o varias extensiones:

- Crear un panel único con los tres gráficos anteriores.
- Analizar la evolución mensual del engagement.
- Incorporar la red de seguidores de `conexiones.json` y comparar seguidores declarados con seguidores dentro de la red.
- Generar usuarios sintéticos con Faker y comparar sus distribuciones con las reales.
- Guardar versiones CSV y Parquet del dataset final y comparar tamaño y tipos conservados.

Estas ampliaciones **no forman parte de la entrega obligatoria** del Bloque 6.


---
## Entrega

Este bloque **no tiene funciones en `pipeline.py`** ni tests propios. Para darlo por terminado, entrega el cuaderno ejecutado de arriba abajo y comprueba que has generado estos ficheros:

| Fichero | Ejercicio | Descripción |
|---|---|---|
| `../data/dataset_cierre.parquet` | 3 | Dataset final con usuarios, publicaciones y `engagement_rate` |
| `../output_graficos/engagement_por_ciudad.html` | 5 | Engagement medio por ciudad |
| `../output_graficos/engagement_por_tipo.html` | 5 | Engagement medio por tipo de contenido |
| `../output_graficos/seguidores_vs_engagement.html` | 5 | Seguidores frente a engagement medio |

Además, el cuaderno debe incluir una conclusión breve en la variable `conclusion_breve`.

---
## Cierre del proyecto

Has recorrido el flujo completo de análisis de datos en Python:

```text
Ficheros base (CSV, Excel, JSON)
        ->
Carga y auditoría inicial
        ->
Limpieza mínima
        ->
Dataset combinado
        ->
Métrica de engagement
        ->
Preguntas analíticas
        ->
Gráficos interactivos
        ->
Conclusión final
```

La parte más importante del cierre no es escribir mucho código, sino demostrar que sabes transformar datos en una respuesta razonada.
